####Example Notebook for BPQ_RNA: A Thermodynamically Informed, hybrid Quantum-Classical Framework for RNA 2D Structure Prediction including Chemical Modifications
**Authors:** Nivedita Dutta, Quentin Vicens, and Liwen Shih  
**Github:** [https://github.com/duttan710/BPQ_RNA](https://github.com/duttan710/BPQ_RNA)

### Introductory note
BPQ_RNA is a hybrid quantum-classical pipeline for predicting RNA secondary structures using a natively base-pair-level Quadratic Unconstrained Binary Optimization (QUBO) formulation. By mapping experimental Nearest Neighbor Database (NNDB) free energies directly onto the Hamiltonian, we bypass coarse-grained pre-assembly. This notebook executes the qubo formulation, quantum optimization via QAOA (facilitated by randomized multistart initialization and multireplica sampling), and a classical maturation post-processing layer to resolve the energetic consequences of loops and dangling ends.

#1. Check the set-up and install dependencies

In [ ]:
import sys
print(sys.version)
!uname -m

3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]
x86_64


In [ ]:
!nvidia-smi

Tue Oct 13 00:55:11 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   38C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
# =====================================================================
# 1. ENVIRONMENT SETUP & DEPENDENCY INSTALLATION
# =====================================================================
!pip uninstall -y qiskit qiskit-aer qiskit-aer-gpu qiskit-aer-gpu-cu11 qiskit-optimization qiskit-algorithms docplex cplex

!pip install qiskit==1.4.* qiskit-aer-gpu-cu11==0.17.* qiskit-optimization==0.7.* qiskit-algorithms docplex cplex matplotlib

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.0/664.0 kB 17.5 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.0/57.0 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.7/6.7 MB 45.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.1/18.1 MB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 237.1/237.1 kB 25.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 327.8/327.8 kB 34.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.2/52.2 MB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 38.6/38.6 MB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 24.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 417.9/417.9 MB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 875.6/875.6 kB 52.2 MB/s eta

In [ ]:
import sys
import subprocess
import warnings
warnings.filterwarnings("ignore")


# =====================================================================
# 2. HARDWARE VALIDATION, IMPORTS & SEEDING
# =====================================================================
import qiskit
import qiskit_aer

print("="*55)
print("VERIFYING REPRODUCIBILITY ENVIRONMENT")
print("="*55)
print(f"Python Version: {sys.version.split()[0]}")
print(f"Qiskit Core Version: {qiskit.__version__}")
print(f"Qiskit Aer Version: {qiskit_aer.__version__}")

# Enforce GPU Availability for Colab Users
try:
    gpu_info = subprocess.check_output('nvidia-smi', shell=True).decode('utf-8')
    if 'failed' in gpu_info or 'NVIDIA-SMI has failed' in gpu_info:
        raise Exception()
    print("\n[SUCCESS] NVIDIA GPU detected. Ready for hardware-accelerated QAOA.")
except:
    print("\n[CRITICAL WARNING] No GPU detected!")
    print("Please go to the Colab Menu -> Runtime -> Change runtime type -> Select 'T4 GPU'.")
    print("Running this notebook on a CPU will result in extremely slow execution or crashes.")
print("="*55)

# Core Scientific & Quantum Imports
import numpy as np
import matplotlib.pyplot as plt

from qiskit_algorithms.utils import algorithm_globals
from qiskit_optimization import QuadraticProgram
from qiskit_algorithms.optimizers import COBYLA
from qiskit_algorithms.minimum_eigensolvers import QAOA, NumPyMinimumEigensolver

# Qiskit V2 Primitives and GPU Simulator (Updated for production runtime)
from qiskit_aer import AerSimulator
from qiskit.primitives import BackendSamplerV2
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager

# Set seeds for exact reproducibility
seed = 42
np.random.seed(seed)
algorithm_globals.random_seed = seed

print("\nDependencies successfully loaded and random seeds set (seed = 42).")

VERIFYING REPRODUCIBILITY ENVIRONMENT
Python Version: 3.13.15
Qiskit Core Version: 1.4.6
Qiskit Aer Version: 0.17.2

[SUCCESS] NVIDIA GPU detected. Ready for hardware-accelerated QAOA.

Dependencies successfully loaded and random seeds set (seed = 42).


## 2. Load Experimnetal (empirical) Free energy Parameters (Turner NNDB parameters)
The thermodynamic stability of RNA secondary structure is primarily driven by adjacent base-pair stacking interactions. The BPQ_RNA framework integrates empirical free energy parameters derived from the Nearest Neighbor Database (NNDB), including updated parameters for canonical stacks and chemical modification ($m^6\text{A}$); directly into the quadratic terms of the QUBO formulation. Complex structural thermodynamic penalties, such as loop initiation and dangling ends, are subsequently evaluated during the classical post-processing step to determine the final free energy ($\Delta G^\circ_{37}$).

If you use this notebook please also cite:

    1. Turner, D. H. & Mathews, D. H. (2009). NNDB: The nearest neighbor parameter database for predicting stability of nucleic acid secondary structure. Nucleic Acids Research. 38, D280-D282.
    2. Mittal, A., Turner, D. H., & Mathews, D. H. (2024). NNDB: An Expanded Database of Nearest Neighbor Parameters for Predicting Stability of Nucleic Acid Secondary Structures. Journal of Molecular Biology, 168549.
    3. Kierzek, E., Zhang, X., Watson, R.M. et al. (2022). Secondary structure prediction for RNA sequences including N6-methyladenosine. Nat Commun 13, 1271.

  



In [ ]:
#Download the files
import urllib.request
import os

# Base RAW URL for your specific Gist
gist_base_url = "https://gist.githubusercontent.com/duttan710/1429a1150dcd7d75fb5ca0cf29b92963/raw/"

files_to_download = [
    "parsed_stack_M_rules.txt",
    "rna.loop.dg",
    "parsed_dangle_M_rules.txt"
]

print("Checking for thermodynamic rule files...")
for filename in files_to_download:
    if not os.path.exists(filename):
        print(f" Downloading {filename} from Gist...")
        raw_url = gist_base_url + filename
        try:
            urllib.request.urlretrieve(raw_url, filename)
            print(f"  -> Successfully downloaded {filename}")
        except Exception as e:
            print(f"  -> Failed to download {filename}. Error: {e}")
    else:
        print(f" [OK] {filename} already exists locally.")

Checking for thermodynamic rule files...
  -> Successfully downloaded parsed_stack_M_rules.txt
  -> Successfully downloaded rna.loop.dg
  -> Successfully downloaded parsed_dangle_M_rules.txt


#### RUN BPQ_RNA pipeline for the test sequences (or sequences of your or sequences interest prefereably requiring 20 qubits)

*   ###### The benchmark sequences are listed (you can uncomment the last 15 sequences and run the pipeline for all 20 sequences or for any of the sequences)
*   ###### Note: when you input a sequence that will require more than 20 qubits, the output will show a warning about this and you can choose not to run the test for that sequence to prevent the exhaustion of the GPU-quota







In [ ]:
import ast
import re
import warnings
import numpy as np
from docplex.mp.model import Model

from qiskit_optimization.translators import from_docplex_mp
from qiskit_optimization.converters import LinearInequalityToPenalty
from qiskit_algorithms import QAOA
from qiskit_algorithms.optimizers import COBYLA
from qiskit_optimization.algorithms import MinimumEigenOptimizer

# Qiskit V2 Primitives and GPU Simulator
from qiskit_aer import AerSimulator
from qiskit.primitives import BackendSamplerV2
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit_algorithms.minimum_eigensolvers import NumPyMinimumEigensolver

warnings.filterwarnings("ignore")

# =====================================================================
# 1. FILE PARSING MODULE
# =====================================================================
def load_stack_rules(filepath):
    try:
        with open(filepath, 'r') as file:
            return ast.literal_eval(file.read())
    except FileNotFoundError:
        print(f"Error: Could not find stack rules file at {filepath}.")
        return {}

def parse_loop_penalties(filepath):
    hairpin_penalties = {}
    try:
        with open(filepath, 'r') as file:
            for line in file:
                if line.startswith('#') or line.startswith('SIZE') or line.startswith('-') or line.strip() == '':
                    continue
                parts = re.split(r'\s+', line.strip())
                if len(parts) >= 4:
                    try:
                        size = int(parts[0])
                        hairpin_val = parts[3]
                        if hairpin_val != '.':
                            hairpin_penalties[size] = float(hairpin_val)
                    except ValueError:
                        continue
        return hairpin_penalties
    except FileNotFoundError:
        print(f"Error: Could not find loop penalties file at {filepath}.")
        return {}

def load_dangle_rules(filepath):
    try:
        with open(filepath, 'r') as file:
            return ast.literal_eval(file.read())
    except FileNotFoundError:
        print(f"Error: Could not find dangle rules file at {filepath}.")
        return {}

# =====================================================================
# 2. SEQUENCE & QUBO MODEL BUILDER
# =====================================================================
def build_rna_qubo(sequence, stack_rules):
    print(f"\nBuilding QUBO Model for Sequence: {sequence}")
    N = len(sequence)
    valid_pairs = [('A', 'U'), ('U', 'A'), ('C', 'G'), ('G', 'C'), ('G', 'U'), ('U', 'G'), ('M', 'U'), ('U', 'M')]
    candidates = [(i, j) for i in range(N) for j in range(i + 4, N) if (sequence[i], sequence[j]) in valid_pairs]

    qp_model = Model(name="RNA_Stacking_Only")
    x = {u: qp_model.binary_var(name=f"x_{u[0]}_{u[1]}") for u in candidates}
    objective = 0

    for idx, u in enumerate(candidates):
        for v in candidates[idx+1:]:
            i, j = u
            k, l = v

            # Stacking Rewards ONLY (No dynamic loop penalties in QUBO)
            if k == i + 1 and l == j - 1:
                stack_key = (sequence[i], sequence[k], sequence[l], sequence[j])
                reverse_key = (sequence[l], sequence[j], sequence[i], sequence[k])
                energy = stack_rules.get(stack_key, stack_rules.get(reverse_key, 0.0))

                if energy < 0:
                    objective += energy * x[u] * x[v]

            # Structural Constraints (Basic clashes)
            if i == k or i == l or j == k or j == l or (i < k < j < l):
                qp_model.add_constraint(x[u] + x[v] <= 1, ctname=f"clash_{i}_{j}_{k}_{l}")

    # Microscopic mathematical gradient to break symmetry softly
    for u in candidates:
        objective += -0.01 * x[u]

    qp_model.minimize(objective)
    qp = from_docplex_mp(qp_model)
    converter = LinearInequalityToPenalty(penalty=50.0)
    qubo = converter.convert(qp)

    print(f"Model created successfully with {qubo.get_num_vars()} qubits.")
    return qubo, candidates

# =====================================================================
# 3. QUANTUM EXECUTION & CONVERGENCE TRACKING (TOP-K MODIFIED)
# =====================================================================
### 1.  Quantum Solver (Returns Top 5 Candidates)
def run_quantum_solver(qubo, num_cycles=12, top_k=5):
    print("\nInitializing Native GPU-Accelerated QAOA solver...")

    # --- i. CLASSICAL (Thermodynamic) BENCHMARK ---
    exact_mes = NumPyMinimumEigensolver()
    exact_result = MinimumEigenOptimizer(exact_mes).solve(qubo)
    exact_fval = exact_result.fval
    print(f" [Ground Truth] Exact Classical QUBO Minimum: {exact_fval:.2f}")

    # --- ii. QAOA SETUP ---
    gpu_backend = AerSimulator(method='statevector', device='GPU')
    gpu_sampler = BackendSamplerV2(backend=gpu_backend)
    pass_manager = generate_preset_pass_manager(optimization_level=1, backend=gpu_backend)

    reps = 4
    qaoa_mes = QAOA(
        sampler=gpu_sampler,
        optimizer=COBYLA(maxiter=500),
        reps=reps,
        transpiler=pass_manager
    )
    qaoa = MinimumEigenOptimizer(qaoa_mes)

    all_samples = []

    print(f"Solving on T4 GPU (Running {num_cycles} multistart cycles)...")
    for cycle in range(num_cycles):
        random_angles = np.random.uniform(0, 2 * np.pi, 2 * reps)
        qaoa_mes.initial_point = random_angles

        current_result = qaoa.solve(qubo)
        print(f"  Cycle {cycle + 1} Raw Quantum Objective: {current_result.fval:.2f}")

        # Collect all samples from this cycle's distribution
        if hasattr(current_result, 'samples') and current_result.samples:
            all_samples.extend(current_result.samples)
        else:
            all_samples.append(current_result)

        # --- EARLY STOPPING CHECK ---
        # If the quantum solver hits the classical minimum (within a small tolerance), break.
        if abs(current_result.fval - exact_fval) < 1e-4:
            print(f"  [Early Stopping] Global minimum ({exact_fval:.2f}) found! Halting remaining cycles.")
            break

    # --- iii. DEDUPLICATION & RANKING ---
    unique_samples = {}
    for s in all_samples:
        bitstr = tuple(s.x)
        # Store the lowest fval found for this specific bitstring
        if bitstr not in unique_samples or s.fval < unique_samples[bitstr].fval:
            unique_samples[bitstr] = s

    # Sort all unique states by objective value and take top_k
    ranked_samples = sorted(unique_samples.values(), key=lambda x: x.fval)[:top_k]

    return ranked_samples, exact_fval
# =====================================================================
# 4. CLASSICAL POST-PROCESSING (FOR TOP-K BATCHING)
# =====================================================================
def analyze_results(sequence, candidates, sample, rank, stack_rules, loop_rules, dangle_rules):
    print("\n" + "-"*55)
    print(f"=== RANK {rank} CANDIDATE RESULTS ===")
    print("-"*55)

    N = len(sequence)
    formed_pairs = []

    # Map the binary array back to candidate base pairs
    for idx, val in enumerate(sample.x):
        if val > 0.5:
            formed_pairs.append(candidates[idx])

    stacking_energy = 0.0
    for idx, u in enumerate(formed_pairs):
        for v in formed_pairs[idx+1:]:
            i, j = u
            k, l = v
            # Check for contiguous stacking
            if k == i + 1 and l == j - 1:
                stack_key = (sequence[i], sequence[k], sequence[l], sequence[j])
                reverse_key = (sequence[l], sequence[j], sequence[i], sequence[k])
                stacking_energy += stack_rules.get(stack_key, stack_rules.get(reverse_key, 0.0))

    dot_bracket = ["."] * N
    for (i, j) in formed_pairs:
        dot_bracket[i] = "("
        dot_bracket[j] = ")"
    structure_str = "".join(dot_bracket)

    print(f"Sequence:                   {sequence}")
    print(f"Quantum Predicted Fold:     {structure_str}")
    print(f"Base Pairs Formed:          {formed_pairs}")
    print(f"Raw Quantum Stacking Core:  {stacking_energy:.2f} kcal/mol")

    if not formed_pairs:
        print("\nNo stable stem structures formed. Final Energy: 0.00 kcal/mol")
        return

    # Loop Penalties
    innermost_pair = min(formed_pairs, key=lambda p: p[1] - p[0])
    loop_size = (innermost_pair[1] - innermost_pair[0]) - 1
    hairpin_penalty = loop_rules.get(loop_size, 0.0)

    print(f"Innermost Stem Pair:        {innermost_pair} -> Loop Size: {loop_size} nts")
    print(f"NNDB Hairpin Penalty:       +{hairpin_penalty:.2f} kcal/mol")

    # Dangle Stabilizations
    outermost_pair = max(formed_pairs, key=lambda p: p[1] - p[0])
    i_out, j_out = outermost_pair
    base_5, base_3 = sequence[i_out], sequence[j_out]

    dangle_5_energy = 0.0
    dangle_3_energy = 0.0

    if i_out > 0:
        dangle_base_5 = sequence[i_out - 1]
        key_5 = (base_5, base_3, dangle_base_5, '5_prime')
        dangle_5_energy = dangle_rules.get(key_5, 0.0)
        if dangle_5_energy < 0:
             print(f"5' Dangle Stabilization ({dangle_base_5} on {base_5}-{base_3}): {dangle_5_energy:.2f} kcal/mol")

    if j_out < N - 1:
        dangle_base_3 = sequence[j_out + 1]
        key_3 = (base_5, base_3, dangle_base_3, '3_prime')
        dangle_3_energy = dangle_rules.get(key_3, 0.0)
        if dangle_3_energy < 0:
             print(f"3' Dangle Stabilization ({dangle_base_3} on {base_5}-{base_3}): {dangle_3_energy:.2f} kcal/mol")

    final_dg = stacking_energy + hairpin_penalty + dangle_5_energy + dangle_3_energy

    print("=" * 55)
    print(f"PREDICTED FREE ENERGY (ΔG°37):      {final_dg:.2f} kcal/mol")
    print("=" * 55)


if __name__ == "__main__":

    # --- COMPLETE BENCHMARK DATASET (S1 - S20) ---
    TEST_SEQUENCES = {
        "S1":  "GCGGAAACGC",       # 10 nt
        "S2":  "GCCAAAAACGG",      # 11 nt
        "S3":  "GGUGUGAACACC",     # 12 nt
        "S4":  "GGUGUGAACMCC",     # 12 nt (m6A Mod)
        "S5":  "GCCGAAAAACGGC",    # 13 nt
        #"S6":  "CACAAAAAAUGUG",    # 13 nt
        #"S7":  "GCGUUAACUCGCA",    # 13 nt
        #"S8":  "GCGGUGAAAUGCC",    # 13 nt
        #"S9":  "GGUGCAUAGCACC",    # 13 nt
        #"S10": "UGACUUUUAAUCA",    # 13 nt
        #"S11": "CACGAAAAAACGUG",   # 14 nt
        #"S12": "GCMGAAAAAACUGC",   # 14 nt (m6A Mod)
        #"S13": "CMCMMMMMMMUGUG",   # 14 nt (m6A Mod)
        #"S14": "ACACGAAAAAACGUG",  # 15 nt
        #"S15": "ACACGAAAAAACGUGC", # 16 nt
        #"S16":  "CCCUUACAAGG",    # 11 nt
        #"S17":  "CCCUUACMAGG",    # 11 nt m6A Mod)
        #"S18": "GCGCAUCGGCGC",    # 12 nt
        #"S19": "GCGCMUCGGCGC",    # 12 nt m6A Mod)
        #"S20":  "UCCCUUACMAGGA",  #13 nt m6A Mod)
    }

    STACK_FILE = "/content/parsed_stack_M_rules.txt"
    LOOP_FILE = "/content/rna.loop.dg"
    DANGLE_FILE = "/content/parsed_dangle_M_rules.txt"

    CYCLES_PER_SEQ = 20
    REPLICATES = 5
    TOP_K = 5

    print("Loading NNDB Parameters for Batch Run...")
    stack_parameters = load_stack_rules(STACK_FILE)
    loop_parameters = parse_loop_penalties(LOOP_FILE)
    dangle_parameters = load_dangle_rules(DANGLE_FILE)

    if stack_parameters and loop_parameters and dangle_parameters:
        for seq_id, seq in TEST_SEQUENCES.items():
            print(f"\n\n{'#'*65}")
            print(f"### BATCH JOB {seq_id}: Length {len(seq)} nt")
            print(f"{'#'*65}")

            qubo, candidates = build_rna_qubo(seq, stack_parameters)

            if qubo.get_num_vars() > 20:
                print(f"\n[WARNING] Qubit count ({qubo.get_num_vars()}) is crossing the classical simulation memory wall!")

            for rep in range(1, REPLICATES + 1):
                print(f"\n{'='*55}")
                print(f"--- RUNNING REPLICATE {rep}/{REPLICATES} FOR {seq_id} ---")
                print(f"{'='*55}")

                top_5_samples, exact_fval = run_quantum_solver(
                    qubo,
                    num_cycles=CYCLES_PER_SEQ,
                    top_k=TOP_K
                )

                if top_5_samples:
                    for rank, sample in enumerate(top_5_samples, start=1):
                        analyze_results(
                            seq,
                            candidates,
                            sample,
                            rank,
                            stack_parameters,
                            loop_parameters,
                            dangle_parameters
                        )
    else:
        print("[ERROR] Failed to load one or more thermodynamic parameter files. Please check file paths.")

Loading NNDB Parameters for Batch Run...


#################################################################
### BATCH JOB S1: Length 10 nt
#################################################################

Building QUBO Model for Sequence: GCGGAAACGC
Model created successfully with 7 qubits.

--- RUNNING REPLICATE 1/5 FOR S1 ---

Initializing Native GPU-Accelerated QAOA solver...
 [Ground Truth] Exact Classical QUBO Minimum: -5.83
Solving on T4 GPU (Running 20 multistart cycles)...
  Cycle 1 Raw Quantum Objective: -3.42
  Cycle 2 Raw Quantum Objective: -0.02
  Cycle 3 Raw Quantum Objective: -5.83
  [Early Stopping] Global minimum (-5.83) found! Halting remaining cycles.

-------------------------------------------------------
=== RANK 1 CANDIDATE RESULTS ===
-------------------------------------------------------
Sequence:                   GCGGAAACGC
Quantum Predicted Fold:     (((....)))
Base Pairs Formed:          [(0, 9), (1, 8), (2, 7)]
Raw Quantum Stacking Core:  -5.80 kcal/mol


##### Extract the 2D structures for each sequence (or your sequence of interest) for the 5 replicas

###### You can copy and paste the output from previous cell to results.txt and use that to extract the 2D structures and the associated enregy information

In [ ]:
import re
import pandas as pd

def parse_results_to_table(filepath):
    print(f"Parsing {filepath}...")

    seq_pattern = re.compile(r"### BATCH JOB (S\d+):")
    rep_pattern = re.compile(r"--- RUNNING REPLICATE (\d+)/\d+ FOR")
    rank_pattern = re.compile(r"=== RANK (\d+) CANDIDATE RESULTS ===")
    fold_pattern = re.compile(r"Quantum Predicted Fold:\s+([().]+)")
    energy_pattern = re.compile(r"PREDICTED FREE ENERGY \(ΔG°37\):\s+([-0-9.]+) kcal/mol")

    data = []

    # State trackers
    current_seq = None
    current_rep = None
    current_rank = None
    current_fold = None

    with open(filepath, 'r') as file:
        for line in file:
            # 1. Catch Sequence ID
            seq_match = seq_pattern.search(line)
            if seq_match:
                current_seq = seq_match.group(1)

            # 2. Catch Replicate Number
            rep_match = rep_pattern.search(line)
            if rep_match:
                current_rep = int(rep_match.group(1))

            # 3. Catch Rank
            rank_match = rank_pattern.search(line)
            if rank_match:
                current_rank = int(rank_match.group(1))

            # 4. Catch Predicted Fold
            fold_match = fold_pattern.search(line)
            if fold_match:
                current_fold = fold_match.group(1)

            # 5. Catch Final Energy & Save Record
            # Since Energy is the last item printed per rank, we append to our list here
            energy_match = energy_pattern.search(line)
            if energy_match:
                energy = float(energy_match.group(1))
                data.append({
                    "Sequence": current_seq,
                    "Replicate": current_rep,
                    "Rank": current_rank,
                    "Predicted Fold": current_fold,
                    "ΔG°37 (kcal/mol)": energy
                })

    # Convert to a DataFrame for easy viewing and sorting
    df = pd.DataFrame(data)

    # Sort logically by Sequence number, then Replicate, then Rank
    # We extract the integer from "S1", "S2" to sort properly (so S10 comes after S9)
    df['Seq_Num'] = df['Sequence'].str.extract(r'(\d+)').astype(int)
    df = df.sort_values(by=['Seq_Num', 'Replicate', 'Rank']).drop(columns=['Seq_Num'])

    return df

# --- RUN THE PARSER ---
# Make sure your file is named exactly 'results.txt' and is in your working directory
parsed_df = parse_results_to_table("/content/results.txt")

# Save to CSV so you can open it in Excel/Sheets
parsed_df.to_csv("compiled_qaoa_results.csv", index=False)

# Display the first 15 rows to verify it worked
print("\n--- Preview of Parsed Data ---")
print(parsed_df.head(15).to_string(index=False))

Parsing /content/results.txt...

--- Preview of Parsed Data ---
Sequence  Replicate  Rank Predicted Fold  ΔG°37 (kcal/mol)
      S1          1     1     (((....)))              -0.2
      S1          1     2     ((.(...)))               2.0
      S1          1     3     ((......))               2.0
      S1          1     4     .((....)).               2.4
      S1          1     5     (.(....).)               5.6
      S1          2     1     (((....)))              -0.2
      S1          2     2     ((.(...)))               2.0
      S1          2     3     ((......))               2.0
      S1          2     4     .((....)).               2.4
      S1          2     5     (.(....).)               5.6
      S1          3     1     (((....)))              -0.2
      S1          3     2     ((.(...)))               2.0
      S1          3     3     ((......))               2.0
      S1          3     4     .((....)).               2.4
      S1          3     5     ..((...).)           